# Bicycle production using a background database

Let's see how we would build a bicycle production foreground system on a background database. We can use the recently released [BAFU 2025 database](https://nexus.openlca.org/database/BAFU), which is freely available.

This is a bit challenging to import right now, so we have a prepared project with an _imperfect_ import already on the server.

This means we won't be woking in the `Bicycle example` anymore, and won't have any of our previously prepared databases.

We start by restoring the BAFU project:

In [ ]:
import bw2io as bi

In [ ]:
bi.restore_project_directory(
    "/etc/data/bafu-2025.tar.gz", 
    project_name="BAFU 2025", 
    overwrite_existing=True
)

Just restoring the project doesn't activate it - this still needs to be done manually.

In [ ]:
import bw2data as bd

In [ ]:
bd.projects.set_current("BAFU 2025")

Check to make sure we have some databases:

In [ ]:
bd.databases

This project uses the ecoinvent 3.12 set of biosphere flows, but doesn't have the ecoinvent database. It also has the 3.12 LCIA impact categories:

In [ ]:
len(bd.methods)

## Exploring the existing bicycle production

This BAFU database has electric and normal bicycles:

In [ ]:
db = bd.Database("BAFU 2025")

In [ ]:
db.search("bicycle")

Let's look at the bill of materials for a bicycle:

In [ ]:
bicycle = bd.get_node(name='Bicycle, at regional storage')

In [ ]:
list(bicycle.technosphere())

We can also filter the BOM to look at metals:

In [ ]:
[
    exc
    for exc in bicycle.technosphere()
    if (
        ("steel" in exc.input["name"].lower()) 
        or ("aluminium" in exc.input["name"].lower())
    )
]

OK - let's only change the amount of metals used. This is way to heavy for the bicycle we want.

## Modification option 1: Via `.copy()` 

There are several ways to make such changes. One thing we could do is to copy the bicycle to a new database, and then change the copied exchange amounts.

In [ ]:
database_with_copy = bd.Database("With copy of bicycle")
database_with_copy.register()

This part is a little tricky - the `database` is the string _label_ of the new database.

In [ ]:
bicycle_copy = bicycle.copy(database=database_with_copy.name)

In [ ]:
bicycle_copy['database']

This has copied over our exchanges:

In [ ]:
len(bicycle_copy.exchanges())

Let's reduce the amount of metals. We can create a filter function for exchanges to get only metal-related ones:

In [ ]:
def is_metals_exchange(exc):
    return (
        ("steel" in exc.input["name"].lower()) 
        or ("aluminium" in exc.input["name"].lower())
    )

And then loop over the exchanges. We can reduced the metals amount by 80%.

In [ ]:
for exc in bicycle_copy.technosphere():
    if is_metals_exchange(exc):
        old_amount = exc["amount"]
        new_amount = old_amount * 0.2
        exc["amount"] = new_amount
        exc.save()
        print(f"Changed amount from {old_amount} to {new_amount} in exchange {exc}")

## Modification option 2: Via `.new_node()`

We can also create a new node from a fresh start.

In [ ]:
clean_database = bd.Database("New database ")
clean_database.register()

In [ ]:
new_bicycle = clean_database.new_node(
    name="bicycle",
    unit="number",
    location="Paris"
)

We can then add some edges to this clean bicycle. This can be completely from our own research, but we can also pull some edges from the existing bicycle definition:

In [ ]:
import random

In [ ]:
for other_exc in bicycle_copy.technosphere():
    if random.random() < 0.4:
        new_bicycle.new_exchange(
            amount=other_exc["amount"],
            input=other_exc.input,
            type=other_exc["type"],
        ).save()

In [ ]:
len(bicycle_copy.exchanges()), len(new_bicycle.exchanges())

# Exercise

Make a copy of a BAFU electric bicycle, and reduce the total amount of steel used to 2.5 kilograms.